# Lab 06-11 · Practical DC Circuits and Measurement

Companion to chapter [06-11](../../chapters/06-11-practical-dc-circuits-and-measurement.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import os
import shutil
import subprocess
import tempfile
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)


def ngspice_values(netlist_path):
    # Run ngspice in batch mode and collect every 'name = value' line it echoes.
    out = subprocess.run(["ngspice", "-b", netlist_path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) == 2:
            try:
                vals[parts[0].strip()] = float(parts[1])
            except ValueError:
                pass
    return vals

ngspice available: True


## Part 1 — Loading curves

A voltmeter of input resistance $R_{in}$ on a source of Thévenin resistance $R_{th}$ reads low by $R_{th}/(R_{th} + R_{in})$.

**🔮 Predict:** at what source resistance does a 10 MΩ meter read 1% low? A 10 GΩ meter?

In [2]:
Rth = np.logspace(1, 10, 400)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for Rin, lab in ((10e6, "10 MΩ meter"), (10e9, "10 GΩ meter")):
    ax.loglog(Rth, 100 * Rth / (Rth + Rin), label=lab)
ax.axhline(1, color="0.4", ls="--", lw=0.8)
ax.set_xlabel("source Thévenin resistance (Ω)"); ax.set_ylabel("reading low by (%)")
ax.set_ylim(1e-6, 100); ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.show()

for Rin in (10e6, 10e9):
    print(f"R_in = {Rin:.0e} Ω: 1% low at R_th = {Rin / 99:.4g} Ω")
assert abs(10e6 / 99 - 101.01e3) < 1

if HAVE_NGSPICE:
    ng = ngspice_values("measure.cir")
    print("ngspice, Example 06-11.1:", ng["ex1b"], "V (10 MΩ),", ng["ex1d"], "V (10 GΩ)")
    assert abs(ng["ex1b"] - 5 * 10 / 10.5) < 1e-4
else:
    print("ngspice not found; see SETUP.md. Skipping the simulator check.")

R_in = 1e+07 Ω: 1% low at R_th = 1.01e+05 Ω
R_in = 1e+10 Ω: 1% low at R_th = 1.01e+08 Ω
ngspice, Example 06-11.1: 4.7619 V (10 MΩ), 4.99975 V (10 GΩ)


A 10 MΩ meter is within 1% for sources up to about 100 kΩ; beyond 1 MΩ it is useless without correction. The 10 GΩ mode moves every number up a thousandfold.

## Part 2 — Choosing the range

A hypothetical 6000-count meter with accuracy ±(0.05% + 1) has ranges of 0.6 V, 6 V and 60 V (resolution 0.1 mV, 1 mV and 10 mV).

**🔮 Predict:** where on each range is the relative uncertainty worst, and how bad is it at 10% of range?

In [3]:
ranges = [(0.6, 1e-4), (6.0, 1e-3), (60.0, 1e-2)]
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for full, res in ranges:
    v = np.linspace(0.02 * full, full, 300)
    ax.semilogx(v, 100 * (0.0005 * v + res) / v, label=f"{full:g} V range")
ax.set_xlabel("reading (V)"); ax.set_ylabel("uncertainty (% of reading)")
ax.set_ylim(0, 1.0); ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.show()

at_10pct = 100 * (0.0005 * 0.6 + 1e-3) / 0.6        # 0.6 V is 10% of the 6 V range (one count = 1 mV)
print(f"at 10% of a range: {100 * (0.0005 + 1 / 600):.3f}%; at full scale: {100 * (0.0005 + 1 / 6000):.4f}%")
assert abs(at_10pct - 100 * (0.0005 + 1 / 600)) < 1e-12

at 10% of a range: 0.217%; at full scale: 0.0667%


Each curve is a hyperbola plus a floor: the count term $1/N$ dominates low on a range. The sawtooth across ranges is why autoranging meters switch up near full scale and down near 10%.

## Part 3 — Kelvin taps against misplaced taps, over temperature

Example 06-11.5: a 1 mΩ shunt with ±250 ppm/°C, and a misplaced tap that adds 0.05 mΩ of copper (+3930 ppm/°C) at each end.

**🔮 Predict:** at what temperature rise does the misplaced-tap reading drift 5% from its calibration at 25 °C?

In [4]:
dT = np.linspace(0, 100, 201)
kelvin = 100 * 250e-6 * dT
misplaced = 100 * (1e-3 * 250e-6 + 0.1e-3 * 3.93e-3) * dT / 1.1e-3
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(dT, kelvin, label="Kelvin taps (worst-case TCR)")
ax.plot(dT, misplaced, label="misplaced taps")
ax.axhline(5, color="0.4", ls="--", lw=0.8)
ax.set_xlabel("temperature rise (°C)"); ax.set_ylabel("drift from 25 °C calibration (%)")
ax.grid(True, alpha=0.3); ax.legend()
plt.show()

slope = (1e-3 * 250e-6 + 0.1e-3 * 3.93e-3) / 1.1e-3
dT5 = 0.05 / slope
print(f"misplaced taps reach 5% at a rise of {dT5:.1f} °C; Kelvin taps are then at {100 * 250e-6 * dT5:.2f}%")

if HAVE_NGSPICE:
    net = '''* shunt over temperature
.options TNOM=25
I5 0 x1 DC 20
R5c1 x1 k1 0.05m tc1=3.93e-3
R5 k1 k2 1m tc1=250e-6
R5c2 k2 0 0.05m tc1=3.93e-3
.control
dc temp 25 125 25
let rk = (v(k1)-v(k2))/20
let rm = v(x1)/20
let dk = (rk[4]/rk[0]-1)*100
let dm = (rm[4]/rm[0]-1)*100
echo "dk = $&dk"
echo "dm = $&dm"
.endc
.end
'''
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "shunt.cir")
        with open(path, "w") as f:
            f.write(net)
        ng = ngspice_values(path)
    print(f"ngspice, 100 °C rise: Kelvin {ng['dk']:.4g}%, misplaced {ng['dm']:.4g}%")
    assert abs(ng["dm"] - 100 * slope * 100) < 1e-3 * 100 * slope * 100

misplaced taps reach 5% at a rise of 85.5 °C; Kelvin taps are then at 2.14%
ngspice, 100 °C rise: Kelvin 2.5%, misplaced 5.845%


The misplaced taps drift 0.058% per °C, more than twice the shunt's own worst case, and reach 5% at a rise of about 86 °C, a temperature a power board can easily see. Calibration at one temperature cannot remove a drift.

## Part 4 — A 48 V bus monitor by Monte Carlo

Problem 06-11.9's divider (140 kΩ over 10 kΩ, each uniform within ±1%) feeds an ADC uniform within ±0.5% of reading. Loading is corrected and not modeled here.

**🔮 Predict:** the worst case is about ±2.4%. What fraction of units exceeds ±1.5%?

In [5]:
rng = np.random.default_rng(611)
N = 200_000
R1 = 140e3 * (1 + rng.uniform(-0.01, 0.01, N))
R2 = 10e3 * (1 + rng.uniform(-0.01, 0.01, N))
adc = 1 + rng.uniform(-0.005, 0.005, N)
err = 100 * ((R2 / (R1 + R2)) * 15 * adc - 1)
print(f"std {err.std():.3f}%; beyond ±1.5%: {100 * np.mean(np.abs(err) > 1.5):.2f}% of units; "
      f"largest seen {np.abs(err).max():.2f}% (worst case +2.39%/−2.34%)")
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.hist(err, bins=120, density=True)
for x in (-1.5, 1.5):
    ax.axvline(x, color="0.4", ls="--", lw=0.8)
ax.set_xlabel("error of the inferred bus voltage (%)"); ax.set_ylabel("density")
plt.show()
assert 5.5 < 100 * np.mean(np.abs(err) > 1.5) < 7.0

std 0.815%; beyond ±1.5%: 6.24% of units; largest seen 2.38% (worst case +2.39%/−2.34%)


About 6% of units fall outside ±1.5%, and essentially none reach the worst case: a typical spread (standard deviation about 0.81%) is far tighter than the guarantee. A product specified at ±1.5% would need testing and rejection or calibration, or better resistors.

## Your turn

1. In Part 1's ammeter analogue, what circuit resistance makes a 1.8 Ω ammeter read 1% low?
2. In Part 3, use a 5 mΩ shunt (±35 ppm/°C) with the same 0.1 mΩ of misplaced copper. What is the effective temperature coefficient?
3. In Part 4, use ±0.1% resistors. What fraction of units exceeds ±1.5% now?

<details><summary>Answers</summary>

1. $R_A/(R_{th} + R_A) = 0.01$ gives $R_{th} = 99\times1.8 = 178.2$ Ω.
2. $(5\times35 + 0.1\times3930)/5.1 = 111$ ppm/°C: still three times the shunt's own figure, though far better than the 1 mΩ case, because the copper is now a smaller fraction of the total.
3. None: the worst case is ±0.69%, so no unit can exceed ±1.5%.

</details>